### 12 - Parsing .csv for Specific Speaker

In this notebook, we will process parallel data to select dialect-specificsentences from 1 speaker.

In [1]:
import pandas as pd
from fst_runtime.fst import Fst
from cg3_process import tokenize, fst_parse_sentence

In [6]:
FST_PATH = "../data/fst/ojibwe.att"
fst = Fst(FST_PATH)

In [5]:
FILENAME = "../data/parallel_data/raw/example_sentences.csv"
dataset = pd.read_csv(FILENAME, on_bad_lines="skip")
print("Row counts =", len(dataset))
dataset.head()

Row counts = 4874


,Ojibwe,English,Speaker,Audio Link
0,Odaanaang bimibatoowan odayan gaa-bimaagonebizod.,The snowmobiler's dog is running behind him.,nj,https://s3.amazonaws.com/ojibwe-audio-transcod...
1,Mishawagaam waasaashkaa.,There are whitecaps out in the lake.,nj,https://s3.amazonaws.com/ojibwe-audio-transcod...
2,Gichi-onzaamaanimad. Waasaashkaa iwe zaaga'igan.,We have a heavy wind. The lake is full of whit...,es,https://s3.amazonaws.com/ojibwe-audio-transcod...
3,Gii-nameshin a'aw ginebig o'omaa gii-pimi-ayaa...,The trail of the snake shows it must have pass...,es,https://s3.amazonaws.com/ojibwe-audio-transcod...
4,Oshkiinamoog gaa-gii-pimi-miikanaakewaad. Gana...,There are fresh tracks of people making a (sno...,nj,https://s3.amazonaws.com/ojibwe-audio-transcod...


In [10]:
ojibwe_list = []
english_list = []
fst_readings_list = []
speaker_list = []

max_lines = len(dataset) # process entire dataset


for i in range(max_lines):
    print(f"[Found Sentence Count = {len(ojibwe_list)}]. Processing sentence #{i+1} / {max_lines} = {(i+1)*100/max_lines:.0f}% ...", end="\r")
    speaker = dataset.iloc[i]["Speaker"]
    if speaker == "nj":
        ojibwe_sentence = dataset.iloc[i]["Ojibwe"]
        ojibwe_list.append(ojibwe_sentence)

        english_sentence = dataset.iloc[i]["English"]
        english_list.append(english_sentence)

        tokens = tokenize(ojibwe_sentence)
        fst_outputs = fst_parse_sentence(tokens, fst)
        fst_readings_list.append(fst_outputs)

        speaker_list.append(speaker)
        
    
print()
print("Number of sentence w/ NJ as speaker =", len(ojibwe_list))

[Found Sentence Count = 1824]. Processing sentence #4874 / 4874 = 100% ...
Number of sentence w/ NJ as speaker = 1825


In [11]:
# create a pandas dataframe and write to output
output_dataset = pd.DataFrame(
    {"ojibwe": ojibwe_list,
     "english": english_list, 
     "fst_readings": fst_readings_list,
     "speaker": speaker_list
     }
)
print("Count =", len(output_dataset))
output_dataset.head()

Count = 1825


,ojibwe,english,fst_readings,speaker
0,Odaanaang bimibatoowan odayan gaa-bimaagonebizod.,The snowmobiler's dog is running behind him.,"[{'word_form': 'odaanaang', 'fst_analyses': ['...",nj
1,Mishawagaam waasaashkaa.,There are whitecaps out in the lake.,"[{'word_form': 'mishawagaam', 'fst_analyses': ...",nj
2,Oshkiinamoog gaa-gii-pimi-miikanaakewaad. Gana...,There are fresh tracks of people making a (sno...,"[{'word_form': 'oshkiinamoog', 'fst_analyses':...",nj
3,Niwii-pabaamaakwii noopimiing. Niwii'-andawago...,I'm going to walk around the woods and set som...,"[{'word_form': 'niwii-pabaamaakwii', 'fst_anal...",nj
4,Dabasaakosin aazhogan Gichi-mookomaanakiing ga...,The bridge going across to the US is a low one...,"[{'word_form': 'dabasaakosin', 'fst_analyses':...",nj


In [12]:
print(output_dataset.iloc[0])

ojibwe          Odaanaang bimibatoowan odayan gaa-bimaagonebizod.
english              The snowmobiler's dog is running behind him.
fst_readings    [{'word_form': 'odaanaang', 'fst_analyses': ['...
speaker                                                        nj
Name: 0, dtype: object


In [13]:
# write to output csv file
OUTPUT_FILENAME = "../data/parallel_data/raw/NJ_speaker_sentences.csv"
print(f"Writing to file {OUTPUT_FILENAME}")
output_dataset.to_csv(OUTPUT_FILENAME, index=False)

Writing to file ../data/parallel_data/raw/NJ_speaker_sentences.csv
